# Topic embeddings, dimensionality reduction, clustering, representation and time plotting with BERTopic

## Set corpus type and configure whether to load existing embeddings and/or existing BERTopic model along with existing topics and probabilities or compute them

In [13]:
corpus_type = "title"
corpus_type = "concat"

load_embeddings = True

load_bertopic_model = True

## Import dask and set up client for resource dashboard

In [ ]:
import numpy as np

from dask import dataframe as dd
from dask.distributed import Client # only for dashboard

client = Client() # only for dashboard
print(client.dashboard_link) # only for dashboard

## Read data set

In [15]:
df = dd.read_parquet(f"artifacts/Preprocessing/arxiv-metadata-cleaned_BERTopic.parquet")
# print(df.dtypes)

## Read 'title' column into 'corpus' column OR concatenate 'title' and 'abstract' into 'corpus' column

In [16]:
if corpus_type == "title":
    df["corpus"] = df["title"]
    print("Using title as corpus.")
elif corpus_type == "concat":
    df["corpus"] = df["title"] + ". " + df["abstract"]
    print("Using combined title and abstract as corpus.")

df = df.drop(columns=["title", "abstract"])
print("Dropped original 'title' and 'abstract' columns.")

# print(df.head())

Using combined title and abstract as corpus.
Dropped original 'title' and 'abstract' columns.


## Embeddings

### Read titles from DataFrame to be used by SentenceTransformer to generate embeddings

In [17]:
print("Computing corpus...")
corpus = df["corpus"].compute().astype(str).tolist()
len_corpus = len(corpus)
output_file_suffix = f"{corpus_type}-{len_corpus}"
print(f"Corpus type: {corpus_type}, number of documents: {len_corpus}")

Computing corpus...
Corpus type: concat, number of documents: 1632186


In [18]:
import pandas as pd
with pd.option_context("display.max_colwidth", None):
    display(df.head())
print(corpus[:5])
print(len(corpus))

,id,update_date,categories,corpus
3929,1807.00987,2020-02-04,gr-qc,"dynamical analysis of brans-dicke universe with inverse power-law effective potential. we study brans-dicke cosmology with an inverse power-law effective potential. by using dynamical analyses, we search for fixed points corresponding to the radiation-like matter and dark energy-dominated era of our universe, and the stability of fixed points is also investigated. we find phase space trajectories which are attracted to the stable point of the dark energy-dominated era from unstable fixed points like matter-dominated era of the universe. the dark energy comes from effective potentials of the brans-dicke field, whose variation (related to the time-variation of the gravitational coupling constant) is shown to be in good agreement with observational data."
5835,1905.08607,2019-05-22,cs.CV eess.IV,"toporesnet: a hybrid deep learning architecture and its application to skin lesion classification. skin cancer is one of the most common cancers in the united states. as technological advancements are made, algorithmic diagnosis of skin lesions is becoming more important. in this paper, we develop algorithms for segmenting the actual diseased area of skin in a given image of a skin lesion, and for classifying different types of skin lesions pictured in a given image. the cores of the algorithms used were based in persistent homology, an algebraic topology technique that is part of the rising field of topological data analysis (tda). the segmentation algorithm utilizes a similar concept to persistent homology that captures the robustness of segmented regions. for classification, we design two families of topological features from persistence diagrams---which we refer to as { persistence statistics} (ps) and { persistence curves} (pc), and use linear support vector machine as classifiers. we also combined those topological features, ps and pc, into resnet-101 model, which we call { toporesnet-101}, the results show that ps and pc are effective in two folds---improving classification performances and stabilizing the training process. although convolutional features are the most important learning targets in cnn models, global information of images may be lost in the training process. because topological features were extracted globally, our results show that the global property of topological features provide additional information to machine learning models."
3412,1906.10602,2019-06-26,cs.DC,"pyramid: a general framework for distributed similarity search. similarity search is a core component in various applications such as image matching, product recommendation and low-shot classification. however, single machine solutions are usually insufficient due to the large cardinality of modern datasets and stringent latency requirement of on-line query processing. we present pyramid, a general and efficient framework for distributed similarity search. pyramid supports search with popular similarity functions including euclidean distance, angular distance and inner product. different from existing distributed solutions that are based on kd-tree or locality sensitive hashing (lsh), pyramid is based on hierarchical navigable small world graph (hnsw), which is the state of the art similarity search algorithm on a single machine. to achieve high query processing throughput, pyramid partitions a dataset into sub-datasets containing similar items for index building and assigns a query to only some of the sub-datasets for query processing. to provide the robustness required by production deployment, pyramid also supports failure recovery and straggler mitigation. pyramid offers a set of concise api such that users can easily use pyramid without knowing the details of distributed execution. experiments on large-scale datasets show that pyramid produces quality results for similarity search, achieves high query processing throughput and is robust under node failure and straggler."
8686,191

['dynamical analysis of brans-dicke universe with inverse power-law effective potential. we study brans-dicke cosmology with an inverse power-law effective potential. by using dynamical analyses, we search for fixed points corresponding to the radiation-like matter and dark energy-dominated era of our universe, and the stability of fixed points is also investigated. we find phase space trajectories which are attracted to the stable point of the dark energy-dominated era from unstable fixed points like matter-dominated era of the universe. the dark energy comes from effective potentials of the brans-dicke field, whose variation (related to the time-variation of the gravitational coupling constant) is shown to be in good agreement with observational data.', 'toporesnet: a hybrid deep learning architecture and its application to skin lesion classification. skin cancer is one of the most common cancers in the united states. as technological advancements are made, algorithmic diagnosis of s

### Generate or load SentenceTransformer embeddings to be used by BERTopic

In [7]:
# Fine-tuning for cpu cores and batch sizes
# import time

# from sentence_transformers import SentenceTransformer

# for cores in (1, 8, 14):
#     print(f"Cores: {cores}")
#     for batch_size in (32, 64, 128, 256):
#         print(f"  Batch size: {batch_size}")
#         model = SentenceTransformer("all-MiniLM-L6-v2")
#         pool = model.start_multi_process_pool(target_devices=["cpu" for _ in range(cores)])
#         tick = time.time()
#         embeddings = model.encode(
#             corpus[:10000],
#             batch_size=batch_size,
#             show_progress_bar=True,
#             pool=pool
#         )
#         tack = time.time()
#         print(f"  Time: {tack - tick:.2f} s")
#         model.stop_multi_process_pool(pool)
#         print("-" * 25)
#     print("-" * 50)

In [8]:
import multiprocessing
import torch
from sentence_transformers import SentenceTransformer

if load_bertopic_model:
    print("Not loading or computing embeddings as trained BERTopic model will be loaded.")
else:
    if load_embeddings:
        print("Loading existing embeddings...")
        embeddings = np.load(f"artifacts/BERTopic/bertopic-{output_file_suffix}-embeddings.npy")
    else:
        print("Computing embeddings...")
        model = SentenceTransformer("all-MiniLM-L6-v2")
        cuda_device_count = torch.cuda.device_count()
        if cuda_device_count > 0:
            print(f"{cuda_device_count} CUDA devices found, generating embeddings via CUDA...")
            embeddings = model.encode(
                corpus,
                batch_size=64,
                show_progress_bar=True
            )
        else:
            print(f"No CUDA devices found, generating embeddings on CPU. This will should take around 4-5 hours...")
            pool = model.start_multi_process_pool(target_devices=["cpu" for _ in range(multiprocessing.cpu_count() - 1)])
            embeddings = model.encode(
                corpus,
                batch_size=64,
                show_progress_bar=True,
                pool=pool
            )
            model.stop_multi_process_pool(pool)


Loading existing embeddings...


### Save embeddings to file

In [9]:
if not load_bertopic_model and not load_embeddings:
    np.save(f"artifacts/BERTopic/bertopic-{output_file_suffix}-embeddings.npy", embeddings)

## Dimensionality Reduction, Clustering and Topic Representation

### Initialize and fit BERTopic model on titles using embeddings or load existing model along with topics and probabilities

In [19]:
import time

import nltk
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

from custom_stopwords import custom_stopwords

nltk.download('stopwords')
stop_words = set(nltk.corpus.stopwords.words('english'))
stop_words = list(stop_words.union(custom_stopwords))

if load_bertopic_model:
    print("Loading existing BERTopic model...")
    topic_model = BERTopic.load(f"artifacts/BERTopic/bertopic-{output_file_suffix}.model")
    print("Loading existing topics and probabilities...")
    topics = np.load(f"artifacts/BERTopic/bertopic-{output_file_suffix}-topics.npy", allow_pickle=True)
    probs = np.load(f"artifacts/BERTopic/bertopic-{output_file_suffix}-probabilities.npy", allow_pickle=True)
else:
    print("Initializing BERTopic model...")
    topic_model = BERTopic(
        verbose=True,
        embedding_model=None, # embeddings are generated/loaded separately above
        umap_model=UMAP(
            n_components=5, # default 2
            min_dist=0.0,
            metric="cosine",
            random_state=2718
        ),
        hdbscan_model=HDBSCAN(
            min_cluster_size=3000,
            min_samples=300,
            prediction_data=True
        ),
        vectorizer_model=CountVectorizer(
            stop_words=stop_words,
            ngram_range=(1,2),
            min_df=10, # ignore terms that appear in fewer than 10 documents
            max_df=0.9, # ignore terms that appear in more than 90% of documents
        )
    )
    print("Fitting BERTopic model to generate topics and probabilities...")
    tick = time.time()
    topics, probs = topic_model.fit_transform(corpus, embeddings)
    tack = time.time()
    with open(f"artifacts/BERTopic/bertopic-{output_file_suffix}-training-time.txt", "w") as f:
        f.write(f"{(tack - tick) / 60:.2f} minutes")

Loading existing BERTopic model...


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/azureuser/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loading existing topics and probabilities...


In [20]:
feature_names = topic_model.vectorizer_model.get_feature_names_out()
print(f"Number of feature names: {len(feature_names)}")

Number of feature names: 674087


### Print general information about found topics

In [21]:
print(len(topic_model.get_topic_info()))
# with pd.option_context(
#                         "display.max_rows", None,
#                         # "display.max_colwidth", None
#                     ):
#     display(topic_model.get_topic_info())

all_topics = topic_model.get_topics()

with open(f"artifacts/BERTopic/bertopic-{output_file_suffix}-topics.csv", "w") as f:
    f.write("topic_id;words\n")

# print all topics and their top 10 words
for topic in range(len(all_topics)-1): # -1 to skip outliers
    topic_words = [word for word, _ in topic_model.get_topic(topic)[:10]]
    print(f"Topic {topic}: {topic_words}")
    with open(f"artifacts/BERTopic/bertopic-{output_file_suffix}-topics.csv", "a") as f:
        f.write(f"{topic};")
        f.write(', '.join(topic_words))
        f.write("\n")

76
Topic 0: ['stars', 'galaxies', 'dark', 'stellar', 'galaxy', 'emission', 'gravitational', 'black hole', 'hole', 'dark matter']
Topic 1: ['algebras', 'varieties', 'cohomology', 'polynomials', 'elliptic', 'projective', 'abelian', 'fractional', 'moduli', 'invariants']
Topic 2: ['wireless', 'mimo', 'beamforming', 'ris', 'antenna', '5g', 'intelligent', '6g', 'csi', 'transmit']
Topic 3: ['particles', 'liquid', 'shear', 'turbulent', 'turbulence', 'droplet', 'polymer', 'reynolds', 'droplets', 'fluids']
Topic 4: ['segmentation', 'mri', 'clinical', 'ct', 'medical image', 'cancer', 'surgical', 'tumor', 'image segmentation', 'brain']
Topic 5: ['speech', 'audio', 'speaker', 'music', 'asr', 'emotion', 'speech recognition', 'voice', 'acoustic', 'speakers']
Topic 6: ['policy', 'reinforcement learning', 'robot', 'rl', 'robots', 'robotic', 'policies', 'reward', 'planning', 'offline']
Topic 7: ['vertex', 'coloring', 'subgraph', 'bipartite', 'planar', 'chromatic', 'hypergraphs', 'clique', 'hypergraph', 

### Save BERTopic model along with found topics and probabilities

In [13]:
if not load_bertopic_model:
    print("Saving BERTopic model...")
    topic_model.save(f"artifacts/BERTopic/bertopic-{output_file_suffix}.model")
    print("Saving topics and probabilities...")
    np.save(f"artifacts/BERTopic/bertopic-{output_file_suffix}-topics.npy", topics)
    np.save(f"artifacts/BERTopic/bertopic-{output_file_suffix}-probabilities.npy", probs)

2026-05-24 12:40:03,202 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Saving BERTopic model...
Saving topics and probabilities...


## Evaluation

### Prepare dictionary and tokenized corpus for coherence computation

In [14]:
import gensim

cleaned_corpus = topic_model._preprocess_text(corpus) 
print(f"CLEANED CORPUS with {len(cleaned_corpus)} documents:")
print(cleaned_corpus[:5])

vectorizer = topic_model.vectorizer_model
analyzer = vectorizer.build_analyzer()

words = vectorizer.get_feature_names_out()

print(f"{len(words)} WORDS")
print(words[:20])

tokens = [analyzer(doc) for doc in cleaned_corpus]
print(f"TOKENS from {len(tokens)} documents:")
print(len(tokens))
print(tokens[:5])

dictionary = gensim.corpora.Dictionary(tokens)
print("DICTIONARY:")
print(dictionary)

tokenized_corpus = [dictionary.doc2bow(token) for token in tokens]
print(f"TOKENIZED CORPUS with {len(tokenized_corpus)} documents:")
print(tokenized_corpus[:5])

topic_words = []
for topic in range(len(set(topics))-topic_model._outliers):
    words = list(zip(*topic_model.get_topic(topic)))[0]
    words = [word for word in words if word in dictionary.token2id]
    topic_words.append(words)
topic_words = [words for words in topic_words if len(words) > 0]
print(f"WORDS in {len(topic_words)} topics:")
print(topic_words[:5])

CLEANED CORPUS with 1632186 documents:
['dynamical analysis of bransdicke universe with inverse powerlaw effective potential we study bransdicke cosmology with an inverse powerlaw effective potential by using dynamical analyses we search for fixed points corresponding to the radiationlike matter and dark energydominated era of our universe and the stability of fixed points is also investigated we find phase space trajectories which are attracted to the stable point of the dark energydominated era from unstable fixed points like matterdominated era of the universe the dark energy comes from effective potentials of the bransdicke field whose variation related to the timevariation of the gravitational coupling constant is shown to be in good agreement with observational data', 'toporesnet a hybrid deep learning architecture and its application to skin lesion classification skin cancer is one of the most common cancers in the united states as technological advancements are made algorithmic

### Compute coherence and save to file

In [ ]:
coherence_model = gensim.models.CoherenceModel(
    topics=topic_words, # list of list of str
    texts=tokens, # tokenized texts
    corpus=tokenized_corpus, # corpus in BoW format
    dictionary=dictionary,
)

print("Calculating coherence...")
coherence = coherence_model.get_coherence()

print(f'Coherence: {coherence}')
with open(f"artifacts/BERTopic/bertopic-{output_file_suffix}.coherence", "w") as f:
    f.write(f'{coherence}\n')

Calculating coherence...


Coherence: 0.8274816819933455


## Topics over time

### Read dates from DataFrame to be used for finding topics over time

In [16]:
print("Computing dates...")
dates = df["update_date"].compute().astype(str).tolist()

Computing dates...


### Find and display topics over time

In [17]:
print("Calculating topics over time...")
topics_over_time = topic_model.topics_over_time(
    docs=corpus,
    timestamps=dates,
    topics=topics,
    datetime_format="%Y-%m-%d",
    nr_bins=20
)

Calculating topics over time...


20it [25:43, 77.20s/it]


In [18]:
display(topics_over_time)

# print("Unique timestamps in topics_over_time:")
# print(topics_over_time["Timestamp"].unique())

,Topic,Words,Frequency,Timestamp
0,-1,"spin, video, electron, social, thermal",15791,2018-12-29 08:05:16.800
1,0,"stars, galaxies, dark, stellar, galaxy",10428,2018-12-29 08:05:16.800
2,1,"algebras, varieties, polynomials, cohomology, ...",6729,2018-12-29 08:05:16.800
3,2,"wireless, mimo, 5g, antenna, beamforming",1495,2018-12-29 08:05:16.800
4,3,"particles, shear, liquid, turbulence, turbulent",1209,2018-12-29 08:05:16.800
...,...,...,...,...
1514,70,"mds, reedsolomon, hamming, dna, repair",311,2025-12-04 20:24:00.000
1515,71,"hamiltonian, orbits, integrable, hamiltonian s...",298,2025-12-04 20:24:00.000
1516,72,"nonhermitian, hall, quantum hall, chern, skin",378,2025-12-04 20:24:00.000
1517,73,"community detection, centrality, hypergraphs, ...",258,2025-12-04 20:24:00.000


### Save topics over time to CSV

In [19]:
topics_over_time.to_csv(f"artifacts/BERTopic/bertopic-{output_file_suffix}-topics_over_time.csv", index=False)

## Visualization

### Visualize topics over time

In [ ]:
fig = topic_model.visualize_topics_over_time(topics_over_time)
fig.write_html(f"artifacts/BERTopic/bertopic-{output_file_suffix}-topics_over_time.html")

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time)

In [22]:
topic_model.visualize_hierarchy()

In [23]:
topic_model.visualize_barchart(top_n_topics=20)

In [24]:
topic_model.visualize_heatmap(width=1000, height=800)

In [25]:
topic_model.visualize_term_rank()

## Run optionally to shutdown Dask client

In [26]:
client.close()